# Module 3 — 02: Preprocessing — Indicators of Heart Disease (2022)



## 1. Download Data

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.copy('/content/drive/MyDrive/kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)
print('Kaggle credentials ready.')

Mounted at /content/drive
Kaggle credentials ready.


In [2]:
!pip install -q --upgrade kaggle
!mkdir -p /content/kaggle_data
!kaggle datasets download -d kamilpytlak/personal-key-indicators-of-heart-disease -p /content/kaggle_data --unzip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.2/126.2 kB 4.5 MB/s eta 0:00:00
Dataset URL: https://www.kaggle.com/datasets/kamilpytlak/personal-key-indicators-of-heart-disease
License(s): CC0-1.0
100% 21.4M/21.4M [00:00<00:00, 164MB/s]



In [3]:
import glob
for f in sorted(glob.glob('/content/kaggle_data/**/*', recursive=True)):
    print(f)

/content/kaggle_data/2020
/content/kaggle_data/2020/heart_2020_cleaned.csv
/content/kaggle_data/2022
/content/kaggle_data/2022/heart_2022_no_nans.csv
/content/kaggle_data/2022/heart_2022_with_nans.csv


In [4]:
import pandas as pd

data_path = '/content/kaggle_data/2022/heart_2022_with_nans.csv'
df = pd.read_csv(data_path)
print('Loaded file:', data_path)
print('Shape:', df.shape)
df.head()

Loaded file: /content/kaggle_data/2022/heart_2022_with_nans.csv
Shape: (445132, 40)


,State,Sex,GeneralHealth,PhysicalHealthDays,MentalHealthDays,LastCheckupTime,PhysicalActivities,SleepHours,RemovedTeeth,HadHeartAttack,...,HeightInMeters,WeightInKilograms,BMI,AlcoholDrinkers,HIVTesting,FluVaxLast12,PneumoVaxEver,TetanusLast10Tdap,HighRiskLastYear,CovidPos
0,Alabama,Female,Very good,0.0,0.0,Within past year (anytime less than 12 months ...,No,8.0,NaN,No,...,NaN,NaN,NaN,No,No,Yes,No,"Yes, received tetanus shot but not sure what type",No,No
1,Alabama,Female,Excellent,0.0,0.0,NaN,No,6.0,NaN,No,...,1.60,68.04,26.57,No,No,No,No,"No, did not receive any tetanus shot in the pa...",No,No
2,Alabama,Female,Very good,2.0,3.0,Within past year (anytime less than 12 months ...,Yes,5.0,NaN,No,...,1.57,63.50,25.61,No,No,No,No,NaN,No,Yes
3,Alabama,Female,Excellent,0.0,0.0,Within past year (anytime less than 12 months ...,Yes,7.0,NaN,No,...,1.65,63.50,23.30,No,No,Yes,Yes,"No, did not receive any tetanus shot in the pa...",No,No
4,Alabama,Female,Fair,2.0,0.0,Within past year (anytime less than 12 months ...,Yes,9.0,NaN,No,...,1.57,53.98,21.77,Yes,No,No,Yes,"No, did not receive any tetanus shot in the pa...",No,No


**Result:** The dataset was downloaded directly from Kaggle using the Kaggle API (authenticated with `kaggle.json`, which was read from Google Drive but not re-saved there) and unzipped into the local Colab runtime storage at `/content/kaggle_data` — no data was written back to Drive. The archive contains three files: `2020/heart_2020_cleaned.csv` (legacy 2020 survey), `2022/heart_2022_no_nans.csv` (2022 survey, already cleaned), and `2022/heart_2022_with_nans.csv` (2022 survey, raw with missing values). We load **`heart_2022_with_nans.csv`** on purpose, since it lets us demonstrate a realistic EDA and preprocessing workflow that includes handling missing values, rather than starting from an already-cleaned file. The loaded dataframe has **445,132 rows and 40 columns**, matching the official 2022 BRFSS update of this dataset.

## 2. Preprocessing

This section prepares the raw data for machine learning: cleaning the target variable, handling missing values, encoding categorical features, and scaling numeric features.

In [5]:
df_clean = df.copy()
before_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['HadHeartAttack']).reset_index(drop=True)
after_rows = len(df_clean)
print(f'Rows before dropping missing target: {before_rows}')
print(f'Rows after dropping missing target: {after_rows}')
print(f'Rows dropped: {before_rows - after_rows}')
df_clean['HadHeartAttack'] = df_clean['HadHeartAttack'].map({'Yes': 1, 'No': 0})
print(df_clean['HadHeartAttack'].value_counts())

Rows before dropping missing target: 445132
Rows after dropping missing target: 442067
Rows dropped: 3065
HadHeartAttack
0    416959
1     25108
Name: count, dtype: int64


**Result:** The 3,065 rows (0.69%) with a missing target value were removed, since imputing a label is not appropriate for a classification target — this leaves 442,067 usable rows. The target was also encoded as a binary integer (Yes=1, No=0) for use in ML models. The resulting class distribution is highly imbalanced: 416,959 negatives (94.32%) vs. 25,108 positives (5.68%), which is important to account for later (e.g. with class weighting or resampling) when training a classifier.

In [6]:
numeric_cols = ['PhysicalHealthDays', 'MentalHealthDays', 'SleepHours', 'HeightInMeters', 'WeightInKilograms', 'BMI']

In [7]:
df_clean[numeric_cols] = df_clean[numeric_cols].fillna(df_clean[numeric_cols].median())
categorical_cols = [c for c in df_clean.columns if c not in numeric_cols and c != 'HadHeartAttack']
mode_values = df_clean[categorical_cols].mode().iloc[0]
df_clean[categorical_cols] = df_clean[categorical_cols].fillna(mode_values)
print('Number of numeric columns imputed (median):', len(numeric_cols))
print('Number of categorical columns imputed (mode):', len(categorical_cols))
print('Total missing values remaining:', df_clean.isna().sum().sum())

Number of numeric columns imputed (median): 6
Number of categorical columns imputed (mode): 33
Total missing values remaining: 0


**Result:** All 6 numeric columns were imputed using the column median (robust to outliers), and all 33 categorical columns were imputed using the column mode (most frequent category). After imputation, the dataset has 0 remaining missing values across all 442,067 rows, so the data is now fully complete and ready for encoding.

In [8]:
binary_cols = [c for c in categorical_cols if df_clean[c].nunique() == 2]
multi_cols = [c for c in categorical_cols if df_clean[c].nunique() > 2]
print('Number of binary categorical columns:', len(binary_cols))
print('Number of multi-category categorical columns:', len(multi_cols))
print(df_clean[categorical_cols].nunique().sort_values().to_string())

Number of binary categorical columns: 22
Number of multi-category categorical columns: 11
Sex                           2
HadStroke                     2
HadAngina                     2
PhysicalActivities            2
HadDepressiveDisorder         2
HadCOPD                       2
HadSkinCancer                 2
HadAsthma                     2
HadKidneyDisease              2
HadArthritis                  2
DeafOrHardOfHearing           2
DifficultyErrands             2
DifficultyDressingBathing     2
DifficultyWalking             2
DifficultyConcentrating       2
BlindOrVisionDifficulty       2
HighRiskLastYear              2
FluVaxLast12                  2
PneumoVaxEver                 2
HIVTesting                    2
AlcoholDrinkers               2
ChestScan                     2
CovidPos                      3
LastCheckupTime               4
ECigaretteUsage               4
TetanusLast10Tdap             4
HadDiabetes                   4
SmokerStatus                  4
RemovedTeeth  

**Result:** Of the 33 categorical columns, 22 are binary (mostly Yes/No health-history flags, e.g. HadStroke, HadAngina, AlcoholDrinkers) and 11 have 3–54 categories. The multi-category columns range from CovidPos (3 categories) up to State (54 categories, i.e. all US states/territories in the survey). Binary columns will be label-encoded to 0/1, and the 11 multi-category columns will be one-hot encoded, since they have no inherent order (except AgeCategory and GeneralHealth, which are ordinal but still small enough to one-hot encode safely).

In [9]:
df_clean[binary_cols] = df_clean[binary_cols].apply(lambda col: pd.factorize(col)[0])
df_clean = pd.get_dummies(df_clean, columns=multi_cols, drop_first=False)
print('Shape before one-hot encoding: (442067, 40)')
print('Shape after one-hot encoding:', df_clean.shape)
print('Number of columns added by one-hot encoding:', df_clean.shape[1] - 40)

Shape before one-hot encoding: (442067, 40)
Shape after one-hot encoding: (442067, 133)
Number of columns added by one-hot encoding: 93


**Result:** The 22 binary categorical columns were label-encoded to integers 0/1, and the 11 multi-category columns were one-hot encoded, expanding the dataset from 40 to 133 columns (93 new dummy columns). This is a large but manageable increase, driven mostly by the State column (54 categories). One-hot encoding avoids implying a false numeric order between unrelated categories (e.g. between different states or races), which is important for most ML algorithms.

In [10]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])
print(df_clean[numeric_cols].describe().round(2).to_string())

       PhysicalHealthDays  MentalHealthDays  SleepHours  HeightInMeters  WeightInKilograms        BMI
count           442067.00         442067.00   442067.00       442067.00          442067.00  442067.00
mean                -0.00             -0.00        0.00            0.00              -0.00      -0.00
std                  1.00              1.00        1.00            1.00               1.00       1.00
min                 -0.49             -0.52       -4.05           -7.64              -2.95      -2.65
25%                 -0.49             -0.52       -0.69           -0.70              -0.73      -0.65
50%                 -0.49             -0.52       -0.02           -0.02              -0.10      -0.16
75%                 -0.14             -0.03        0.66            0.75               0.43       0.42
max                  3.00              3.09       11.41            6.82              10.27      11.50


**Result:** The 6 numeric columns were standardized using StandardScaler, so each now has mean ≈ 0 and standard deviation = 1. This puts all numeric features on a comparable scale, which is important for distance-based or gradient-based ML algorithms (e.g. logistic regression, KNN, SVM, neural networks) that would otherwise be dominated by features with larger raw ranges (like WeightInKilograms vs. SleepHours). Note the wide max values for some scaled features (e.g. HeightInMeters max = 6.82, PhysicalHealthDays max = 3.00), reflecting the outliers seen earlier in the EDA step — these are expected and not an error.

In [11]:
print('Final preprocessed dataset shape:', df_clean.shape)
print('Target distribution (%):')
print((df_clean['HadHeartAttack'].value_counts(normalize=True) * 100).round(2))
df_clean.head()

Final preprocessed dataset shape: (442067, 133)
Target distribution (%):
HadHeartAttack
0    94.32
1     5.68
Name: proportion, dtype: float64


,Sex,PhysicalHealthDays,MentalHealthDays,PhysicalActivities,SleepHours,HadHeartAttack,HadAngina,HadStroke,HadAsthma,HadSkinCancer,...,AgeCategory_Age 70 to 74,AgeCategory_Age 75 to 79,AgeCategory_Age 80 or older,"TetanusLast10Tdap_No, did not receive any tetanus shot in the past 10 years","TetanusLast10Tdap_Yes, received Tdap","TetanusLast10Tdap_Yes, received tetanus shot but not sure what type","TetanusLast10Tdap_Yes, received tetanus shot, but not Tdap",CovidPos_No,CovidPos_Tested positive using home test without a health professional,CovidPos_Yes
0,0,-0.491530,-0.515488,0,0.656772,0,0,0,0,0,...,False,False,True,False,False,True,False,True,False,False
1,0,-0.491530,-0.515488,0,-0.687687,0,0,0,0,1,...,False,False,True,True,False,False,False,True,False,False
2,0,-0.258523,-0.154541,1,-1.359916,0,0,0,0,1,...,False,False,False,True,False,False,False,False,False,True
3,0,-0.491530,-0.515488,1,-0.015457,0,0,0,1,0,...,False,False,False,True,False,False,False,True,False,False
4,0,-0.258523,-0.515488,1,1.329001,0,0,0,0,0,...,False,False,False,True,False,False,False,True,False,False


**Result:** The final preprocessed dataset has 442,067 rows and 133 columns, with 0 missing values, a binary numeric target (HadHeartAttack), all categorical features encoded (label encoding for binary columns, one-hot encoding for multi-category columns), and all 6 numeric features standardized. The target remains imbalanced (94.32% No vs. 5.68% Yes), which was preserved intentionally rather than artificially rebalanced, since resampling should typically be applied only to the training split (after a train/test split) to avoid data leakage. This dataset is now ready to be split into training and test sets and used to train a classification model for heart attack risk prediction.